# 気象条件なども追加していきたい！

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
# trainデータの読み込み
df_train = pd.read_csv("/home/keiseki/JR_train_snow/20.Data/train.csv", encoding="cp932")
# 糸魚川駅、富山駅での通常車両の新幹線の車両への着雪量データであるフラグを付与
df_train["回送列車フラグ"] = 0

In [3]:
df_train

,年月日,列車番号,停車駅名,フェンダー部分(東京方向),台車部分,フェンダー部分(金沢方向),合計,回送列車フラグ
0,2016-01-19,3500E,富山,0.0,0.0,0.000000,0.000000,0
1,2016-01-19,562E,富山,0.0,0.0,0.000000,0.000000,0
2,2016-01-19,560E,糸魚川,0.0,0.0,0.000000,0.000000,0
3,2016-01-19,560E,富山,0.0,0.0,0.002986,0.002986,0
4,2016-01-19,558E,糸魚川,0.0,0.0,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...
15310,2016-12-31,554E,糸魚川,0.0,0.0,0.000000,0.000000,0
15311,2016-12-31,574E,糸魚川,0.0,0.0,0.000000,0.000000,0
15312,2016-12-31,576E,富山,0.0,0.0,0.000000,0.000000,0
15313,2016-12-31,558E,富山,0.0,0.0,0.000000,0.000000,0


In [4]:
# 回送列車データの読み込み
df_through = pd.read_csv("/home/keiseki/JR_train_snow/20.Data/out_of_service.csv", encoding="cp932")
# 糸魚川駅、富山駅での回送車両への着雪量データであるフラグを付与
df_through["回送列車フラグ"] = 1

In [5]:
# 通常列車、回送列車のデータを結合
df_all = pd.concat([df_train, df_through], ignore_index=True)

In [6]:
# ダイヤ情報の読み込み
df_dia = pd.read_csv("/home/keiseki/JR_train_snow/20.Data/diagram.csv", encoding="cp932")
# df_dia

In [7]:
df_dia_T = df_dia.T
df_dia_T = df_dia_T.replace('↓', '通過')

df_dia_T.columns = df_dia_T.iloc[0]
df_dia_T = df_dia_T.iloc[1:]

# 通過セルは左右の時刻の中間値に置換し、連続通過は区間を等分する
import importlib.util
from pathlib import Path

module_path = Path("/home/keiseki/JR_train_snow/30.src/utils/utils.py")
spec = importlib.util.spec_from_file_location("jr_utils", module_path)
jr_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(jr_utils)

df_dia_T = jr_utils.fill_pass_time_cells_in_dataframe(df_dia_T)

# df_dia_T

In [8]:
# 列車情報とダイア情報を結合
df_all = pd.merge(df_all, df_dia_T, how='left', left_on='列車番号', right_index=True)
# df_all

In [9]:
# 金沢着雪ゼロ列車
df_zero = pd.read_csv("/home/keiseki/JR_train_snow/20.Data/kanazawa_nosnow.csv", encoding="cp932", header=None, names=["列車番号"])

# df_zeroに記載されている列車番号に「金沢着雪ゼロ列車」フラグを付与
df_zero["金沢着雪ゼロ列車フラグ"] = 1
# df_zero

In [10]:
df_merge = pd.merge(df_all, df_zero, how='left', on='列車番号')
df_merge["金沢着雪ゼロ列車フラグ"] = df_merge["金沢着雪ゼロ列車フラグ"].fillna(0)
df_merge

,年月日,列車番号,停車駅名,フェンダー部分(東京方向),台車部分,フェンダー部分(金沢方向),合計,回送列車フラグ,停車時刻,金沢,新高岡,富山,黒部宇奈月温泉,糸魚川,上越妙高,飯山,長野,金沢着雪ゼロ列車フラグ
0,2016-01-19,3500E,富山,0.000000,0.000000,0.000000,0.000000,0,NaN,6:00,6:10,6:19,6:29,6:38,6:48,6:57,7:07,0.0
1,2016-01-19,562E,富山,0.000000,0.000000,0.000000,0.000000,0,NaN,11:56,12:10,12:19,12:32,12:46,12:59,13:10,13:20,0.0
2,2016-01-19,560E,糸魚川,0.000000,0.000000,0.000000,0.000000,0,NaN,10:56,11:10,11:19,11:32,11:46,11:59,12:11,12:24,1.0
3,2016-01-19,560E,富山,0.000000,0.000000,0.002986,0.002986,0,NaN,10:56,11:10,11:19,11:32,11:46,11:59,12:11,12:24,1.0
4,2016-01-19,558E,糸魚川,0.000000,0.000000,0.000000,0.000000,0,NaN,9:21,9:35,9:45,9:57,10:11,10:25,10:37,11:00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15334,2016-01-25,NaN,糸魚川,0.000048,0.002732,0.003908,0.006688,1,09:25:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
15335,2016-01-25,NaN,糸魚川,0.000221,0.004932,0.005369,0.010521,1,08:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
15336,2016-01-25,NaN,糸魚川,0.000136,0.005393,0.002608,0.008136,1,08:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
15337,2016-01-25,NaN,糸魚川,0.002262,0.017006,0.000122,0.019390,1,07:42:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [11]:
df_merge.to_pickle("/home/keiseki/JR_train_snow/20.Data/train_data_all.pkl")

---
testデータにも同じ特徴量を極力付与

In [12]:
test_path = '../20.Data/test.csv'
df_test = pd.read_csv(test_path, encoding='cp932', index_col=0)

In [13]:
# 列車情報とダイア情報を結合
df_test_all = pd.merge(df_test, df_dia_T, how='left', left_on='列車番号', right_index=True)
# 金沢着雪ゼロ列車のフラグを付与
df_test_merge = pd.merge(df_test_all, df_zero, how='left', on='列車番号')
df_test_merge["金沢着雪ゼロ列車フラグ"] = df_test_merge["金沢着雪ゼロ列車フラグ"].fillna(0)

# pickle出力
df_test_merge.to_pickle("/home/keiseki/JR_train_snow/20.Data/test_data_all(0910).pkl")

---
気象データの読み込みと日付単位への分割

In [14]:
# 読み込み
df_weather = pd.read_csv(
    "/home/keiseki/JR_train_snow/20.Data/weather.csv",
    encoding="cp932",
    index_col=0
)
df_weather = df_weather.reset_index()

# 年月日時をdatetime型にする
df_weather["年月日時"] = pd.to_datetime(df_weather["年月日時"])

# 日付と時刻を分離
df_weather["日付"] = df_weather["年月日時"].dt.date
df_weather["時刻"] = df_weather["年月日時"].dt.strftime("%-H:%M")

# 横持ちする気象項目
value_cols = [
    col for col in df_weather.columns
    if col not in ["年月日時", "日付", "時刻", "地点"]
]

# 日付 × 地点 × 時刻 を横持ち
df_weather_wide = df_weather.pivot(
    index="日付",
    columns=["地点", "時刻"],
    values=value_cols
)

# MultiIndexになったカラムを
# 「地点_項目_時刻」に変換
df_weather_wide.columns = [
    f"{place}_{item}_{time}"
    for item, place, time in df_weather_wide.columns
]

# indexを通常の列に戻す
df_weather_wide = df_weather_wide.reset_index()
# 日付単位の気象データを出力
df_weather_wide.to_pickle("/home/keiseki/JR_train_snow/20.Data/weather_data_daily.pkl")

In [15]:
# 日付単位の気象データをtrain/testデータに結合
df_train_all = pd.read_pickle("/home/keiseki/JR_train_snow/20.Data/train_data_all.pkl")
df_test_all = pd.read_pickle("/home/keiseki/JR_train_snow/20.Data/test_data_all(0910).pkl")
df_weather = pd.read_pickle("/home/keiseki/JR_train_snow/20.Data/weather_data_daily.pkl")

# 年月日をdatetime型に変換
df_train_all["年月日"] = pd.to_datetime(df_train_all["年月日"])
df_test_all["年月日"] = pd.to_datetime(df_test_all["年月日"])
df_weather["日付"] = pd.to_datetime(df_weather["日付"])


df_train_all = pd.merge(df_train_all, df_weather, how='left', left_on='年月日', right_on='日付', indicator=True)
df_test_all = pd.merge(df_test_all, df_weather, how='left', left_on='年月日', right_on='日付', indicator=True)

In [ ]:
import importlib.util
import yaml
from pathlib import Path

utils_path = Path("/home/keiseki/JR_train_snow/30.src/utils/utils.py")
utils_spec = importlib.util.spec_from_file_location("jr_utils", utils_path)
jr_utils = importlib.util.module_from_spec(utils_spec)
utils_spec.loader.exec_module(jr_utils)

with open("/home/keiseki/JR_train_snow/00.config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

pass_weather = config["PASS_TIME_WEATHER_FEATURE"]
station_cols_train = [station for station in pass_weather["stations"] if station in df_train_all.columns]
station_cols_test = [station for station in pass_weather["stations"] if station in df_test_all.columns]

df_train_all = jr_utils.build_pass_time_weather_features(
    df_train_all,
    weather_variables=pass_weather["variables"],
    station_columns=station_cols_train,
    suffix=pass_weather["suffix"],
)
df_test_all = jr_utils.build_pass_time_weather_features(
    df_test_all,
    weather_variables=pass_weather["variables"],
    station_columns=station_cols_test,
    suffix=pass_weather["suffix"],
)

---
積雪計データの読み込みと時間帯の集約

In [16]:
# ==========================================
# ① 読み込み
# ==========================================

df_snow_meter = pd.read_csv(
    "/home/keiseki/JR_train_snow/20.Data/snowfall.csv",
    encoding="cp932",
    index_col=0
)

df_snow_meter = df_snow_meter.reset_index()

# 年月日時をdatetimeに変換
df_snow_meter["年月日時"] = pd.to_datetime(
    df_snow_meter["年月日時"]
)

# 日付を作成
df_snow_meter["日付"] = (
    df_snow_meter["年月日時"].dt.date
)

# ==========================================
# ② 時間帯を作成
# ==========================================

def get_time_zone(hour):
    if hour < 6:
        return "0-6"
    elif hour < 12:
        return "6-12"
    elif hour < 18:
        return "12-18"
    else:
        return "18-24"


df_snow_meter["時間帯"] = (
    df_snow_meter["年月日時"]
    .dt.hour
    .apply(get_time_zone)
)

# ==========================================
# ③ 集約対象の項目
# ==========================================

value_cols = [
    "積雪深(軌道)",
    "積雪深(側溝)",
    "積雪深(軌道A)",
    "積雪深(軌道B)"
]

# ==========================================
# ④ 日付 × 地域 × 時間帯ごとに集約
# ==========================================

df_snow_meter_summary = (
    df_snow_meter
    .groupby(
        ["日付", "地域名", "時間帯"]
    )[value_cols]
    .agg(["mean", "max", "min"])
)

# ==========================================
# ⑤ MultiIndexの列を通常の列名に変換
# ==========================================

df_snow_meter_summary.columns = [
    f"{item}_{stat}"
    for item, stat in df_snow_meter_summary.columns
]

df_snow_meter_summary = (
    df_snow_meter_summary
    .reset_index()
)

# ==========================================
# ⑥ 日付 × 時間帯 × 地域
#    → 地域を横持ち
# ==========================================

df_snow_meter_wide = (
    df_snow_meter_summary
    .set_index(
        ["日付", "時間帯", "地域名"]
    )
    .unstack("地域名")
)

# 地域 × 項目のMultiIndexをフラット化
df_snow_meter_wide.columns = [
    f"{region}_{item}"
    for item, region in df_snow_meter_wide.columns
]

df_snow_meter_wide = (
    df_snow_meter_wide
    .reset_index()
)

# ==========================================
# ⑦ 時間帯を横持ち
# ==========================================

df_snow_meter_wide = (
    df_snow_meter_wide
    .set_index(["日付", "時間帯"])
    .unstack("時間帯")
)

# ==========================================
# ⑧ MultiIndexを完全に解除
# ==========================================

df_snow_meter_wide.columns = [
    f"{item}_{time_zone}"
    for item, time_zone in df_snow_meter_wide.columns
]

df_snow_meter_wide = (
    df_snow_meter_wide
    .reset_index()
)

df_snow_meter_wide["日付"] = pd.to_datetime(df_snow_meter_wide["日付"])

# ==========================================
# ⑨ 確認
# ==========================================

print(df_snow_meter_wide.shape)
print(df_snow_meter_wide.columns.tolist())

display(df_snow_meter_wide.head())

(243, 481)
['日付', '富山野々上_積雪深(軌道)_mean_0-6', '富山野々上_積雪深(軌道)_mean_12-18', '富山野々上_積雪深(軌道)_mean_18-24', '富山野々上_積雪深(軌道)_mean_6-12', '小矢部_積雪深(軌道)_mean_0-6', '小矢部_積雪深(軌道)_mean_12-18', '小矢部_積雪深(軌道)_mean_18-24', '小矢部_積雪深(軌道)_mean_6-12', '布施川_積雪深(軌道)_mean_0-6', '布施川_積雪深(軌道)_mean_12-18', '布施川_積雪深(軌道)_mean_18-24', '布施川_積雪深(軌道)_mean_6-12', '朝日_積雪深(軌道)_mean_0-6', '朝日_積雪深(軌道)_mean_12-18', '朝日_積雪深(軌道)_mean_18-24', '朝日_積雪深(軌道)_mean_6-12', '東金沢_積雪深(軌道)_mean_0-6', '東金沢_積雪深(軌道)_mean_12-18', '東金沢_積雪深(軌道)_mean_18-24', '東金沢_積雪深(軌道)_mean_6-12', '梶屋敷_積雪深(軌道)_mean_0-6', '梶屋敷_積雪深(軌道)_mean_12-18', '梶屋敷_積雪深(軌道)_mean_18-24', '梶屋敷_積雪深(軌道)_mean_6-12', '水橋白岩_積雪深(軌道)_mean_0-6', '水橋白岩_積雪深(軌道)_mean_12-18', '水橋白岩_積雪深(軌道)_mean_18-24', '水橋白岩_積雪深(軌道)_mean_6-12', '津幡_積雪深(軌道)_mean_0-6', '津幡_積雪深(軌道)_mean_12-18', '津幡_積雪深(軌道)_mean_18-24', '津幡_積雪深(軌道)_mean_6-12', '田海川_積雪深(軌道)_mean_0-6', '田海川_積雪深(軌道)_mean_12-18', '田海川_積雪深(軌道)_mean_18-24', '田海川_積雪深(軌道)_mean_6-12', '高岡赤祖父_積雪深(軌道)_mean_0-6', '高岡赤祖父_積雪深(軌道)_mean_12-18', '高岡赤祖父_積雪深(軌道)_

,日付,富山野々上_積雪深(軌道)_mean_0-6,富山野々上_積雪深(軌道)_mean_12-18,富山野々上_積雪深(軌道)_mean_18-24,富山野々上_積雪深(軌道)_mean_6-12,小矢部_積雪深(軌道)_mean_0-6,小矢部_積雪深(軌道)_mean_12-18,小矢部_積雪深(軌道)_mean_18-24,小矢部_積雪深(軌道)_mean_6-12,布施川_積雪深(軌道)_mean_0-6,...,津幡_積雪深(軌道B)_min_18-24,津幡_積雪深(軌道B)_min_6-12,田海川_積雪深(軌道B)_min_0-6,田海川_積雪深(軌道B)_min_12-18,田海川_積雪深(軌道B)_min_18-24,田海川_積雪深(軌道B)_min_6-12,高岡赤祖父_積雪深(軌道B)_min_0-6,高岡赤祖父_積雪深(軌道B)_min_12-18,高岡赤祖父_積雪深(軌道B)_min_18-24,高岡赤祖父_積雪深(軌道B)_min_6-12
0,2015-12-01,-5.914286,-22.027778,-22.194444,-22.555556,-6.0,-23.0,-23.0,-23.0,-6.485714,...,-22.0,-22.0,-22.0,-22.0,-22.0,-22.0,-22.0,-22.0,-22.0,-22.0
1,2015-12-02,-22.194444,-22.027778,-22.027778,-22.611111,-23.0,-23.0,-23.0,-23.0,-23.000000,...,-22.0,-22.0,-22.0,-23.0,-22.0,-23.0,-22.0,-22.0,-22.0,-22.0
2,2015-12-03,-22.861111,-22.555556,-22.361111,-22.944444,-23.0,-23.0,-23.0,-23.0,-23.000000,...,-22.0,-22.0,-22.0,-22.0,-22.0,-22.0,-22.0,-22.0,-22.0,-22.0
3,2015-12-04,-22.111111,-22.277778,-22.444444,-22.333333,-23.0,-23.0,-23.0,-23.0,-23.000000,...,-22.0,-22.0,-22.0,-22.0,-22.0,-23.0,-22.0,-22.0,-22.0,-23.0
4,2015-12-05,-22.555556,-22.527778,-22.555556,-22.527778,-23.0,-23.0,-23.0,-23.0,-23.000000,...,-22.0,-22.0,-22.0,-22.0,-22.0,-23.0,-22.0,-22.0,-22.0,-22.0


In [17]:
df_train_all = pd.merge(df_train_all, df_snow_meter_wide, how='left', left_on='年月日', right_on='日付')
df_test_all = pd.merge(df_test_all, df_snow_meter_wide, how='left', left_on='年月日', right_on='日付')

In [ ]:
import importlib.util
import yaml
from pathlib import Path

utils_path = Path("/home/keiseki/JR_train_snow/30.src/utils/utils.py")
utils_spec = importlib.util.spec_from_file_location("jr_utils", utils_path)
jr_utils = importlib.util.module_from_spec(utils_spec)
utils_spec.loader.exec_module(jr_utils)

with open("/home/keiseki/JR_train_snow/00.config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

pass_weather = config["PASS_TIME_WEATHER_FEATURE"]
station_cols = [station for station in pass_weather["stations"] if station in df_train_all.columns]
test_station_cols = [station for station in pass_weather["stations"] if station in df_test_all.columns]

df_train_all = jr_utils.build_pass_time_weather_features(
    df_train_all,
    weather_variables=pass_weather["variables"],
    station_columns=station_cols,
    suffix=pass_weather["suffix"],
)
df_test_all = jr_utils.build_pass_time_weather_features(
    df_test_all,
    weather_variables=pass_weather["variables"],
    station_columns=test_station_cols,
    suffix=pass_weather["suffix"],
)

In [18]:
df_train_all.columns = (
    df_train_all.columns
    .str.replace(":", "_", regex=False)
    .str.replace("(", "_", regex=False)
    .str.replace(")", "_", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("㎡", "m2", regex=False)
)
df_test_all.columns = (
    df_test_all.columns
    .str.replace(":", "_", regex=False)
    .str.replace("(", "_", regex=False)
    .str.replace(")", "_", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("㎡", "m2", regex=False)
)

---
datetime64方はLightGBMに渡しにくいので、

In [19]:
df_train_all["年月日"] = pd.to_datetime(df_train_all["年月日"])

df_train_all["年"] = df_train_all["年月日"].dt.year
df_train_all["月"] = df_train_all["年月日"].dt.month
df_train_all["日"] = df_train_all["年月日"].dt.day
df_train_all["曜日"] = df_train_all["年月日"].dt.dayofweek

In [20]:
df_test_all["年月日"] = pd.to_datetime(df_test_all["年月日"])

df_test_all["年"] = df_test_all["年月日"].dt.year
df_test_all["月"] = df_test_all["年月日"].dt.month
df_test_all["日"] = df_test_all["年月日"].dt.day
df_test_all["曜日"] = df_test_all["年月日"].dt.dayofweek

In [ ]:
df_train_all.to_pickle("/home/keiseki/JR_train_snow/20.Data/train_time_weather.pkl")
df_test_all.to_pickle("/home/keiseki/JR_train_snow/20.Data/test_time_weather.pkl")

---
学習データを冬季限定にする

In [ ]:
# df_train_winter = df_train_all[df_train_all["月"].isin([11, 12, 1, 2, 3])]

In [ ]:
# df_train_winter.to_pickle("/home/keiseki/JR_train_snow/20.Data/train_winter_only.pkl")
# df_test_all.to_pickle("/home/keiseki/JR_train_snow/20.Data/test_weather_snow.pkl")